# Nigeria in the Prompt, America in the Answer — evaluation run

Measures **cosmetic localization**: whether stating the user's country changes the
institutional world a model's advice presupposes, or only its vocabulary.

Four conditions per item, three models, matched Nigeria/US pairs:

| | |
|---|---|
| **C1** bare | no locale signal — establishes the model's default world |
| **C2** localized | same question, country stated — does the *flag count* move, or only the words? |
| **C3** knowledge probe | the same fact asked directly — separates absent knowledge from inert knowledge |
| **C4-true / C4-false** | user pushes back with a correct, then an invented, claim about local practice |

C4-false is the load-bearing control. If a model capitulates to an invented claim about
Nigerian practice as readily as to a true one, then agreement carries no information, and
what a user experiences as the model learning is the model deferring.

**Runs on a single H200.** All three models are loaded one at a time; nothing here needs
more than ~12 GB at a time.

In [1]:
# %pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

## 1 — Environment

In [2]:
import os
import torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.set_per_process_memory_fraction(0.50, device=0)

print("GPU Ready:", torch.cuda.get_device_name(0))
print("Allocated:", torch.cuda.memory_allocated(0) / 1e9, "GB")

GPU Ready: NVIDIA H200
Allocated: 0.0 GB


In [3]:
!nvidia-smi

Sat Sep 19 21:08:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200                    On  |   00000000:23:00.0 Off |                    0 |
| N/A   63C    P0            618W /  700W |   32629MiB / 143771MiB |     89%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:

# import os
# os.chdir("/home/ubuntu/nanochat/cosmetic-localization")
# print(os.getcwd())


In [5]:
# Pinned so a rerun reproduces. vLLM must be recent: two of the three checkpoints
# are 2026 architectures. If vLLM refuses a model, engine.py falls back to HF
# generate automatically -- slower, irrelevant on an H200.
import sys
print(sys.executable)
# %pip install -U "transformers>=4.57" "accelerate>=1.0" "huggingface_hub>=0.26" pandas

/vol/side_env/bin/python


In [6]:
import subprocess, sys, pathlib, os, json
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
import torch, transformers
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| transformers", transformers.__version__)

name, memory.total [MiB]
NVIDIA H200, 143771 MiB



/vol/side_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.14.0+cu126 | cuda True | transformers 5.17.0


In [7]:
# Point this at the repo. Either clone it or mount it -- the notebook only needs
# the data/, build/, eval/ and score/ directories.
REPO = pathlib.Path(os.environ.get("COSLOC_REPO", "/home/ubuntu/nanochat/cosmetic-localization"))
assert REPO.exists(), f"set COSLOC_REPO or clone the repo to {REPO}"
os.chdir(REPO)
sys.path[:0] = [str(REPO / "eval"), str(REPO / "score")]
print("repo:", REPO)

repo: /home/ubuntu/nanochat/cosmetic-localization


## 2 — Verify the instruments are the frozen ones

The scoring instruments are frozen with a committed hash. Tuning a scoring
instrument after seeing results is the one thing that would make this study
worthless, so the run refuses to start against an unrecorded version.

In [8]:
import hashlib
recorded = {}
for line in (REPO / "data/INSTRUMENT_HASHES.txt").read_text().splitlines():
    if line.startswith("#") or not line.strip():
        continue
    name, _, digest = line.split()[0], None, line.split("sha256=")[1]
    recorded[name] = digest

for name, digest in recorded.items():
    actual = hashlib.sha256((REPO / "data" / name).read_bytes()).hexdigest()
    status = "ok" if actual == digest else "MODIFIED"
    print(f"{status:9s} {name}")
    assert actual == digest, (
        f"{name} does not match the frozen hash. If the change is intended, record it in "
        f"data/CHECKLIST_CHANGELOG.md with a reason and regenerate INSTRUMENT_HASHES.txt.")

!python score/detect.py --self-test

ok        checklist.json
ok        ng_markers.json
ok    74 checks passed


## 3 — Build the run manifest

`PHASE = "gate"` runs the 24 machine-drafted probe pairs, whose only job is to decide
whether the effect is there before any authoring cost is paid. Their numbers never appear
in the paper and `validate_items.py --release` refuses to ship them.

`PHASE = "study"` runs the authored dataset.

In [9]:
PHASE = "gate"      # "gate" | "study"
SAMPLES = 3         # per cell; small models are noisy and one sample per cell is the
                    # easiest thing for a reviewer to distrust
SEED = 0

if PHASE == "gate":
    !python probe/make_drafts.py
    SRC, ITEMS, MANIFEST = "probe/draft_items.jsonl", "probe/draft_items_paired.jsonl", "probe/gate_manifest.jsonl"
else:
    SRC, ITEMS, MANIFEST = "data/items_src.jsonl", "data/items.jsonl", "results/manifest.jsonl"

!python build/render_prompts.py --items {SRC} --out-items {ITEMS} --out-manifest {MANIFEST} --samples {SAMPLES} --seed {SEED}
!python build/validate_items.py --items {ITEMS} {"" if PHASE == "gate" else "--release"}

24 draft pairs -> probe/draft_items.jsonl
  commerce=3, employment=3, finance=3, government=3, healthcare=3, payments=3, tenancy=3, utilities=3
  signalling=8  us_correction=11  law/practice=11
24 authored -> 48 items -> 222 contexts (666 generations per model)
  C1            24
  C2            80
  C3            48
  C4-false      35
  C4-true       35
ok    48 items, 24 pairs [draft]
      gt tiers: t1=29, t2=18, t4=1
      law/practice divergence: 11


## 4 — Smoke test

Two items, one model, one sample. Catches the things that silently ruin a full run:
a chat template that rejects a system turn, Qwen's thinking traces leaking into the
response text, truncation mid-sentence, a refusal.

Read the output. Do not skip this.

In [10]:
from engine import MODELS, RunConfig, run
for k, v in MODELS.items():
    print(f"{k:14s} {v['hf_id']:28s} {v['lab']:18s} {v['origin']}  {v['params']}")

qwen3.5-4b     Qwen/Qwen3.5-4B              Alibaba            CN  4B
gemma4-e2b     google/gemma-4-E2B-it        Google DeepMind    US  5.1B raw / 2.3B effective
minicpm5-2b    openbmb/MiniCPM5-2B          OpenBMB            CN  2.5B


In [11]:
import sys
print("Active Python:", sys.executable)

# Route pip cache and unpack directories to the 3TB drive
%env PIP_CACHE_DIR=/vol/.cache/pip
%env TMPDIR=/vol/tmp

Active Python: /vol/side_env/bin/python
env: PIP_CACHE_DIR=/vol/.cache/pip
env: TMPDIR=/vol/tmp


In [12]:
# %pip install vllm

In [13]:
smoke = RunConfig(models=["qwen3.5-4b"], manifest=REPO / MANIFEST,
                  out=REPO / "results/smoke.jsonl", samples=1, seed=SEED, limit=2)
run(smoke)

import itertools
for line in itertools.islice(open(REPO / "results/smoke.jsonl"), 4):
    r = json.loads(line)
    print(f"\n--- {r['item_id']} {r['condition']} [{r['locale']}] " + "-" * 40)
    print(r["response"][:700])

manifest: 7 contexts x 1 samples x 1 models = 7 generations

=== qwen3.5-4b  (Qwen/Qwen3.5-4B, Alibaba, CN) ===
    vllm unavailable (ModuleNotFoundError: No module named 'vllm'); falling back to HF generate


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 426/426 [00:01<00:00, 220.25it/s]


    backend=hf


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


    s0 C1          1 rows    32.0s
    s0 C2          3 rows    30.8s
    s0 C3          1 rows    29.4s
    s0 C4-true     1 rows    11.0s
    s0 C4-false    1 rows     9.4s

wrote 7 responses -> /home/ubuntu/nanochat/cosmetic-localization/results/smoke.jsonl

--- emp_001 C1 [NG] ----------------------------------------
Congratulations on your upcoming change! While resigning can be an emotional process, leaving a professional legacy is what matters most. Here is a checklist of key items to sort out before your last day to ensure a smooth transition and maintain your professional reputation.

### 1. Knowledge Transfer & Documentation
*   **Create Handover Notes**: Document critical processes, passwords (where appropriate), current project status, and any known bugs or pending issues.
*   **Update Shared Drives/Confluence**: Ensure all files are organized and accessible to your replacement or team members.
*   **Schedule a Final Meeting**: Book a time with your manager and direct repor

## 5 — Full run

Three models, all conditions, three samples. Models load sequentially and the GPU is
released between them.

In [14]:
cfg = RunConfig(models=list(MODELS), manifest=REPO / MANIFEST,
                out=REPO / f"results/responses_{PHASE}.jsonl",
                samples=SAMPLES, seed=SEED, temperature=0.7,
                extra={"gpu_memory_utilization": 0.85})
run(cfg)

manifest: 222 contexts x 3 samples x 3 models = 1998 generations

=== qwen3.5-4b  (Qwen/Qwen3.5-4B, Alibaba, CN) ===
    vllm unavailable (ModuleNotFoundError: No module named 'vllm'); falling back to HF generate


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 426/426 [00:02<00:00, 208.49it/s]


    backend=hf
    s0 C1         24 rows    59.0s
    s0 C2         80 rows   140.1s
    s0 C3         48 rows    66.0s
    s0 C4-true    35 rows    35.6s
    s0 C4-false   35 rows    35.5s
    s1 C1         24 rows    42.2s
    s1 C2         80 rows   108.7s
    s1 C3         48 rows    65.2s
    s1 C4-true    35 rows    35.3s
    s1 C4-false   35 rows    35.4s
    s2 C1         24 rows    41.9s
    s2 C2         80 rows   114.2s
    s2 C3         48 rows    64.9s
    s2 C4-true    35 rows    34.9s
    s2 C4-false   35 rows    34.9s

=== gemma4-e2b  (google/gemma-4-E2B-it, Google DeepMind, US) ===
    vllm unavailable (ModuleNotFoundError: No module named 'vllm'); falling back to HF generate


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1951/1951 [00:02<00:00, 666.81it/s]


    backend=hf
    s0 C1         24 rows    60.5s
    s0 C2         80 rows   109.4s
    s0 C3         48 rows    63.8s
    s0 C4-true    35 rows    33.0s
    s0 C4-false   35 rows    31.5s
    s1 C1         24 rows    46.1s
    s1 C2         80 rows   108.2s
    s1 C3         48 rows    78.4s
    s1 C4-true    35 rows    31.6s
    s1 C4-false   35 rows    32.3s
    s2 C1         24 rows    43.0s
    s2 C2         80 rows   107.6s
    s2 C3         48 rows    64.0s
    s2 C4-true    35 rows    31.3s
    s2 C4-false   35 rows    31.3s

=== minicpm5-2b  (openbmb/MiniCPM5-2B, OpenBMB, CN) ===
    vllm unavailable (ModuleNotFoundError: No module named 'vllm'); falling back to HF generate


/vol/side_env/lib/python3.12/site-packages/huggingface_hub/file_download.py:769: UserWarning: Not enough free disk space to download the file. The expected file size is: 5033.56 MB. The target location /home/ubuntu/.cache/huggingface/hub/models--openbmb--MiniCPM5-2B/blobs only has 4221.50 MB free disk space.
  warnings.warn(


RuntimeError: Task error: File reconstruction error: IO Error: No space left on device (os error 28)

In [29]:
cfg = RunConfig(models=[list(MODELS)[-1]], manifest=REPO / MANIFEST,
                out=REPO / f"results/responses_{PHASE}.jsonl",
                samples=SAMPLES, seed=SEED, temperature=0.7,
                extra={"gpu_memory_utilization": 0.85})
run(cfg)

manifest: 222 contexts x 3 samples x 1 models = 666 generations

=== minicpm5-2b  (openbmb/MiniCPM5-2B, OpenBMB, CN) ===
    vllm unavailable (ModuleNotFoundError: No module named 'vllm'); falling back to HF generate


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 381/381 [00:01<00:00, 334.12it/s]


    backend=hf
    s0 C1         24 rows    46.8s
    s0 C2         80 rows    79.7s
    s0 C3         48 rows    57.4s
    s0 C4-true    35 rows    28.1s
    s0 C4-false   35 rows    27.4s
    s1 C1         24 rows    30.7s
    s1 C2         80 rows    93.0s
    s1 C3         48 rows    48.0s
    s1 C4-true    35 rows    27.6s
    s1 C4-false   35 rows    27.3s
    s2 C1         24 rows    31.0s
    s2 C2         80 rows    81.2s
    s2 C3         48 rows    48.0s
    s2 C4-true    35 rows    33.8s
    s2 C4-false   35 rows    29.1s

wrote 666 responses -> /home/ubuntu/nanochat/cosmetic-localization/results/responses_gate.jsonl


PosixPath('/home/ubuntu/nanochat/cosmetic-localization/results/responses_gate.jsonl')

## 6 — Determinism check

"Fixed seed" should be a verified claim, not a stated one. Re-runs one cell at
temperature 0 and asserts the output is byte-identical.

In [17]:
det = RunConfig(models=["qwen3.5-4b"], manifest=REPO / MANIFEST,
                out=REPO / "results/_det_a.jsonl", samples=1, seed=SEED,
                temperature=0.0, limit=2)
run(det)
det.out = REPO / "results/_det_b.jsonl"
run(det)

a = [json.loads(l)["response"] for l in open(REPO / "results/_det_a.jsonl")]
b = [json.loads(l)["response"] for l in open(REPO / "results/_det_b.jsonl")]
same = sum(x == y for x, y in zip(a, b))
print(f"identical: {same}/{len(a)}")
if same != len(a):
    print("NOT deterministic at temperature 0 -- report this rather than claiming a fixed seed.")

manifest: 7 contexts x 1 samples x 1 models = 7 generations

=== qwen3.5-4b  (Qwen/Qwen3.5-4B, Alibaba, CN) ===
    vllm unavailable (ModuleNotFoundError: No module named 'vllm'); falling back to HF generate


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 426/426 [00:02<00:00, 212.18it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


    backend=hf
    s0 C1          1 rows    29.2s
    s0 C2          3 rows    22.2s
    s0 C3          1 rows    17.9s
    s0 C4-true     1 rows     9.1s
    s0 C4-false    1 rows     9.0s

wrote 7 responses -> /home/ubuntu/nanochat/cosmetic-localization/results/_det_a.jsonl
manifest: 7 contexts x 1 samples x 1 models = 7 generations

=== qwen3.5-4b  (Qwen/Qwen3.5-4B, Alibaba, CN) ===
    vllm unavailable (ModuleNotFoundError: No module named 'vllm'); falling back to HF generate


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 426/426 [00:02<00:00, 209.56it/s]


    backend=hf
    s0 C1          1 rows    18.0s
    s0 C2          3 rows    18.6s
    s0 C3          1 rows    18.0s
    s0 C4-true     1 rows     9.0s
    s0 C4-false    1 rows     9.1s

wrote 7 responses -> /home/ubuntu/nanochat/cosmetic-localization/results/_det_b.jsonl
identical: 7/7


In [30]:
# # Create the target directory on the 3TB volume
# !mkdir -p /vol/huggingface

# # Move any existing HF cache to /vol to save root disk space
# !mv ~/.cache/huggingface/* /vol/huggingface/ 2>/dev/null || true

# # Replace ~/.cache/huggingface with a symlink to /vol
# !rm -rf ~/.cache/huggingface
# !ln -s /vol/huggingface ~/.cache/huggingface

# %env HF_HOME=/vol/huggingface

env: HF_HOME=/vol/huggingface


## 7 — Stage-1 detection

Deterministic. The frozen lexicon emits every checklist hit with its span, its
sentence, and a polarity: `assumed` (the answer builds on the institution) versus
`contrasted` (the answer explicitly says it does not apply here). Only `assumed`
counts toward the flag rate — treating explicit non-transfer as a failure would be
the worst available scoring bug.

No model judges anything. A model scoring locale-appropriateness carries the blind
spot under test.

In [31]:
!python score/detect.py --in results/responses_{PHASE}.jsonl --out results/detected_{PHASE}.jsonl

Traceback (most recent call last):
  File "/home/ubuntu/nanochat/cosmetic-localization/score/detect.py", line 341, in <module>
    raise SystemExit(main())
                     ^^^^^^
  File "/home/ubuntu/nanochat/cosmetic-localization/score/detect.py", line 331, in main
    hits = det.detect(rec.get("response", ""))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ubuntu/nanochat/cosmetic-localization/score/detect.py", line 101, in detect
    sents = split_sentences(text)
            ^^^^^^^^^^^^^^^^^^^^^
  File "/home/ubuntu/nanochat/cosmetic-localization/score/detect.py", line 50, in split_sentences
    assert len(masked) == len(text), "masking must preserve offsets"
           ^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: masking must preserve offsets


## 8 — Preliminary gate read

**These numbers are not results.** They are detector-only, the items are machine-drafted,
and G2 and G3 need hand-coding before they mean anything. This cell answers one question:
is there enough signal to justify authoring 242 items?

In [ ]:
!python analysis/gate_report.py --detected results/detected_{PHASE}.jsonl

## 9 — Export for human scoring

Writes the queues `score/confirm.py` works through: C4 correction codes and C3
correctness in full, C1/C2 flag confirmation on a stratified 25% sample.

Confirming a highlighted span takes about five seconds. Scoring a response cold takes
about forty. That difference is what makes ~12 hours of scoring finishable instead of ~50.

In [ ]:
!python score/confirm.py --build-queues --detected results/detected_{PHASE}.jsonl --out score/queues
!ls -la score/queues

In [1]:
!landscape-sysinfo

  System load:  1.87               Processes:               404
  Usage of /:   68.9% of 71.61GB   Users logged in:         1
  Memory usage: 8%                 IPv4 address for enp1s0: 100.70.57.4
  Swap usage:   0%


In [2]:
!lsblk
!df -h

NAME    MAJ:MIN RM  SIZE RO TYPE MOUNTPOINTS
sr0      11:0    1  366K  0 rom  
vda     253:0    0   75G  0 disk 
├─vda1  253:1    0   74G  0 part /
├─vda14 253:14   0    4M  0 part 
├─vda15 253:15   0  106M  0 part 
└─vda16 259:0    0  913M  0 part 
vdb     253:16   0  2.9T  0 disk /mnt/volume_j2q09m9
Filesystem      Size  Used Avail Use% Mounted on
tmpfs            24G  1.4M   24G   1% /run
efivarfs        256K   26K  225K  11% /sys/firmware/efi/efivars
/dev/vda1        72G   50G   23G  69% /
tmpfs           118G   28K  118G   1% /dev/shm
tmpfs           5.0M     0  5.0M   0% /run/lock
/dev/vdb        2.9T   88G  2.7T   4% /mnt/volume_j2q09m9
tmpfs            24G   24K   24G   1% /run/user/1000
